Last Updated: May 2026(Tested on Google Colab)

In [1]:
!python --version

Python 3.12.13


# IMPORTANT

1. Run the installation cell first
2. Add required API keys in Colab Secrets
3. Run notebook cells sequentially

# If running in Google Colab Please ensure you update below keys in "secrets" on the left and give access to this notebook

1.   OPENAI_API_KEY
2.   GROQ_API_KEY

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [1]:
!pip install -q \
agno==1.4.4 \
openai==1.59.6 \
yfinance==0.2.54 \
pyautogen==0.2.35 \
python-dotenv==1.0.1 \
ipykernel==6.29.5

In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
#If executing from local machine, run below 2 lines to load keys (.env should be present in same directory with keys in it)
# from dotenv import load_dotenv
# load_dotenv()


#If executing from Colab, run below lines to load keys (keys should be added in colab secrets and access given to this notebook)
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

In [3]:
# Configure - note code updated here,
config_list = [{
    "model": "llama-3.3-70b-versatile",
    "api_key": os.environ.get("GROQ_API_KEY"),
    "base_url": "https://api.groq.com/openai/v1",
    "price": [0, 0]
}]

In [4]:
from autogen import ConversableAgent, AssistantAgent, UserProxyAgent
from autogen import initiate_chats
import pprint

agent = ConversableAgent(
    name="chatbot",
    llm_config={"config_list": config_list,"price": [0, 0]},
    human_input_mode="NEVER",
)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [5]:
reply = agent.generate_reply(
    messages=[{"content": "Tell me a joke.", "role": "user"}]
)
# note code updated here,
print(reply)

Here's one:

What do you call a fake noodle?

An impasta.

I hope that one made you laugh. Do you want to hear another?


1. ConversableAgent - ConversableAgents are designed for structured conversations.(example - onboarding)
2. AssistantAgent - AssistantAgent are task oriented agents, often having expertise
3. UserProxyAgent - act as intermediaries between the user and the system. They relay user inputs

In [6]:
# Define the RegistrationAgent for collecting user details
registration_agent = ConversableAgent(
    name="User Registration Agent",
    system_message='''You are a friendly registration assistant.
    Your task is to collect the user's full name, company name, and the product they are interested in.
    Do not ask for additional details. Respond with 'END_SESSION' once all details are obtained.''',
    llm_config={"config_list": config_list},
    code_execution_config=False,
    human_input_mode="NEVER",
)

In [9]:
# Define the HelpDeskAgent for troubleshooting
help_desk_agent = AssistantAgent(
    name="Help Desk Agent",
    system_message='''You are a knowledgeable support assistant.
    Your role is to assist users in buying our product.
    Offer clear steps, share useful resources, and escalate if required.
    Respond with 'END_SESSION' once the issue is addressed.''',
    llm_config={"config_list": config_list},
    code_execution_config=False,
    human_input_mode="NEVER",
)

In [10]:
# Define the SalesAgent for product recommendations
sales_agent = AssistantAgent(
    name="Sales Recommendation Agent",
    system_message='''You are a sales assistant.
    Based on user information, suggest the most suitable product and provide additional details.
    If the user expresses interest, proceed with gathering purchase intent details.
    Respond with 'END_SESSION' when user decides to proceed or not.''',
    llm_config={"config_list": config_list},
    code_execution_config=False,
    human_input_mode="NEVER",
)

In [11]:
# Define the RelayAgent for communication handling
relay_agent = UserProxyAgent(
    name="Interaction Relay Agent",
    system_message='''You act as a bridge between users and specialized agents.
    You do not process requests yourself but ensure smooth communication.''',
    llm_config=False,
    code_execution_config=False,
    human_input_mode="ALWAYS",
)

In [12]:
conversation_flow = [
    {
        "sender": registration_agent,
        "recipient": relay_agent,
        "message": "Hi! I'm here to help with your onboarding. Please provide your full name, company, and the product you're interested in.",
        "summary_method": "reflection_with_llm",
        "summary_args": {
            "summary_prompt": "Extract user details as JSON: {'full_name': '', 'company': '', 'product': ''}",
        },
        "max_turns": 2,
        "clear_history": True
    },
    {
        "sender": relay_agent,
        "recipient": sales_agent,
        "message": "Based on user details, suggest a suitable product.",
        "summary_method": "reflection_with_llm",
        "summary_args": {
            "summary_prompt": "Summarize recommended products as JSON: {'recommended_product': '', 'justification': ''}",
        },
        "max_turns": 3,
        "clear_history": False
    },
    {
        "sender": relay_agent,
        "recipient": help_desk_agent,
        "message": "Based on sale recommendation, closen the conversation with formal sale pitch",
        "summary_method": "reflection_with_llm",
        "summary_args": {
            "summary_prompt": "Summarize the assistance provided as JSON: {'resolution_steps': []}",
        },
        "max_turns": 3,
        "clear_history": False
    }
]

# Execute the conversations
session_results = initiate_chats(conversation_flow)
pprint.pp(session_results)


********************************************************************************
Starting a new chat....

********************************************************************************
User Registration Agent (to Interaction Relay Agent):

Hi! I'm here to help with your onboarding. Please provide your full name, company, and the product you're interested in.

--------------------------------------------------------------------------------
Replying as Interaction Relay Agent. Provide feedback to User Registration Agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation: 

>>>>>>>> NO HUMAN INPUT RECEIVED.

>>>>>>>> USING AUTO REPLY...
Interaction Relay Agent (to User Registration Agent):



--------------------------------------------------------------------------------
User Registration Agent (to Interaction Relay Agent):

To move forward, could you please provide me with your full name?

------------------------------------------------------------------